In [ ]:
import pandas as pd
import numpy as np
import duckdb
import re
from itertools import combinations
from optbinning import OptimalBinning
from joblib import Parallel, delayed
import multiprocessing
from typing import Dict, List, Optional
from prettytable import PrettyTable
import json
import logging


class StrategicSegmentBuilder:
    """
    A combinatorial heuristic engine for extracting highly predictive, mutually exclusive
    segments from tabular data using Optimal Binning and Apriori-style pruning.

    Attributes:
        target (str): The dependent binary variable (e.g., 'default_flag').
        n_jobs (int): Number of CPU cores to utilize for parallel processing.
        min_sample_size (int): Minimum volume required for a segment to be valid.
        min_lift (float): Minimum lift (segment rate / base rate) required.
        top_n_vars (int): Number of top Information Value (IV) features to consider per iteration.
        max_segments (int): The maximum number of mutually exclusive segments to extract.
        enable_diversity (bool): If True, restricts rules from using features from the same group.
        enable_1way (bool): Include 1-dimensional rules in the final output.
        enable_2way (bool): Include 2-dimensional rules in the final output.
        enable_3way (bool): Include 3-dimensional rules in the final output.
        feature_groups (dict): Mapping of analytical categories to specific columns
            (e.g., {'liquidity': ['avg_bal_1m', 'avg_bal_3m'], 'delinquency': ['pay_0', 'pay_2']}).
    """

    def __init__(
        self,
        target: str,
        n_jobs: int = -1,
        min_sample_size: int = 1000,
        min_lift: float = 2.0,
        top_n_vars: int = 20,
        max_segments: int = 10,
        enable_diversity: bool = False,
        enable_1way: bool = True,
        enable_2way: bool = True,
        enable_3way: bool = True,
        feature_groups: Optional[Dict[str, List[str]]] = None,
    ):

        self.target = target
        self.n_jobs = (
            n_jobs if n_jobs != -1 else max(1, multiprocessing.cpu_count() - 1)
        )
        self.min_sample_size = min_sample_size
        self.min_lift = min_lift
        self.top_n_vars = top_n_vars
        self.max_segments = max_segments
        self.segments = []
        self.enable_diversity = enable_diversity
        self.enable_1way = enable_1way
        self.enable_2way = enable_2way
        self.enable_3way = enable_3way
        self.feature_groups = feature_groups or {}

    def _validate_feature_groups(self, df: pd.DataFrame):
        """Validates that all columns provided in feature_groups exist in the dataset."""
        if not self.feature_groups:
            return

        all_df_cols = set(df.columns) - {self.target}
        all_grouped_vars = set()
        for group, vars_list in self.feature_groups.items():
            for var in vars_list:
                if var not in all_df_cols:
                    raise ValueError(
                        f"Validation Error: Variable '{var}' defined in group '{group}' "
                        f"does not exist in the provided DataFrame."
                    )
                all_grouped_vars.add(var)

        print(
            f"Feature group validation successful. {len(all_grouped_vars)} variables validated."
        )

    def get_group(self, var: str) -> str:
        """Identifies the assigned business category for a given feature."""
        for group, vars_list in self.feature_groups.items():
            if var in vars_list:
                return group
        return var

    def is_diverse(self, combo: tuple) -> bool:
        """Validates if a combination of features spans distinct analytical categories."""
        if not self.enable_diversity:
            return True
        groups = [self.get_group(v) for v in combo]
        return len(groups) == len(set(groups))

    def compute_iv_ranking(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculates the Information Value (IV) for all predictors against the target."""

        def _get_iv(col):
            try:
                # FIX: Explicitly added check for 'string' to catch native pandas string types
                dtype = (
                    "categorical"
                    if str(df[col].dtype) in ["object", "category", "string", "str"]
                    else "numerical"
                )
                optb = OptimalBinning(name=col, dtype=dtype)
                optb.fit(df[col].values, df[self.target].values)
                iv = optb.binning_table.build().IV.iloc[-1]
                return {"variable": col, "iv": iv * 100}
            except Exception:
                return {"variable": col, "iv": 0}

        cols = [c for c in df.columns if c != self.target]
        results = Parallel(n_jobs=self.n_jobs)(delayed(_get_iv)(col) for col in cols)
        return (
            pd.DataFrame(results)
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

    def create_binned_df(self, df: pd.DataFrame, variables: List[str]) -> pd.DataFrame:
        """Discretizes continuous variables into optimal categorical bins."""
        binned_df = pd.DataFrame(index=df.index)
        for col in variables:
            # FIX: Explicitly added check for 'string' to catch native pandas string types
            dtype = (
                "categorical"
                if str(df[col].dtype) in ["object", "category", "string", "str"]
                else "numerical"
            )
            optb = OptimalBinning(name=col, dtype=dtype)
            optb.fit(df[col].values, df[self.target].values)

            transformed_bins = optb.transform(df[col], metric="bins")
            binned_df[col] = pd.Categorical(transformed_bins)

        binned_df[self.target] = df[self.target].values
        return binned_df

    def _agg_combinations(
        self, binned_df: pd.DataFrame, combo_list: List[tuple], base_rate: float
    ) -> pd.DataFrame:
        """Parallelized aggregation to test all active combinations for lift and volume thresholds."""

        def _process_combo(combo):
            summary = (
                binned_df.groupby(list(combo), observed=True)
                .agg(count=(self.target, "size"), events=(self.target, "sum"))
                .reset_index()
            )

            summary = summary[summary["count"] >= self.min_sample_size].copy()
            if summary.empty:
                return None

            summary["rate"] = (summary["events"] / summary["count"]) * 100
            summary["lift"] = summary["rate"] / (base_rate * 100)

            summary = summary[summary["lift"] >= self.min_lift]
            if summary.empty:
                return None

            rule_parts = [f"{c}=" + summary[c].astype(str) for c in combo]
            summary["rule"] = rule_parts[0]
            for rp in rule_parts[1:]:
                summary["rule"] += " & " + rp
            summary["combo_vars"] = [combo] * len(summary)

            return summary[["rule", "count", "rate", "lift", "combo_vars"]]

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(_process_combo)(c) for c in combo_list
        )
        valid_results = [r for r in results if r is not None]
        return (
            pd.concat(valid_results, ignore_index=True)
            if valid_results
            else pd.DataFrame()
        )

    def parse_rule_to_sql(self, rule_str: str) -> str:
        """
        Translates the OptBinning interval syntax into a production-ready SQL WHERE clause.
        Handles both numeric intervals and categorical sets (including Pandas StringArray wrappers).
        """
        parts = [p.strip() for p in rule_str.split("&")]
        sql_conditions = []

        for part in parts:
            if "=" not in part:
                continue
            col, interval = part.split("=", 1)
            col, interval = col.strip(), interval.strip()

            # Use regex to find anything inside square brackets [...]
            match_brackets = re.search(r"\[(.*)\]", interval)

            # Determine if this part represents a categorical list
            is_categorical = False
            if match_brackets:
                content = match_brackets.group(1)
                # It's categorical if it contains quotes, text array type wrappers,
                # or doesn't start with standard continuous interval notation like '[' or '('
                if (
                    "'" in interval
                    or '"' in interval
                    or "Array" in interval
                    or "Categorical" in interval
                ):
                    is_categorical = True
                elif not (interval.startswith("[") or interval.startswith("(")):
                    is_categorical = True
                elif (
                    len(content.split(",")) > 2
                ):  # Numeric intervals only ever have 2 elements
                    is_categorical = True

            # 1. Process Categorical Variable Blocks
            if is_categorical:
                content = match_brackets.group(1)
                # Split out individual items and clean up outer quotes
                items = [
                    i.strip().strip("'").strip('"')
                    for i in content.split(",")
                    if i.strip()
                ]

                formatted_parts = []
                for i in items:
                    # Keep raw numbers unquoted, wrap true strings in single quotes
                    if i.replace(".", "", 1).replace("-", "", 1).isdigit():
                        formatted_parts.append(i)
                    else:
                        formatted_parts.append(f"'{i}'")

                formatted_items = ", ".join(formatted_parts)
                sql_conditions.append(f"{col} IN ({formatted_items})")
                continue

            # 2. Process Special/Missing Value Blocks
            if interval in ["Special", "Missing"]:
                sql_conditions.append(f"{col} IS NULL")
                continue

            # 3. Process Continuous Numeric Interval Blocks
            if interval.startswith("[") or interval.startswith("("):
                left_bracket, right_bracket = interval[0], interval[-1]
                content = interval[1:-1]

                if "," in content:
                    lower_str, upper_str = [x.strip() for x in content.split(",", 1)]

                    col_conds = []
                    if lower_str.lower() != "-inf":
                        op = ">=" if left_bracket == "[" else ">"
                        col_conds.append(f"{col} {op} {lower_str}")
                    if upper_str.lower() != "inf":
                        op = "<=" if right_bracket == "]" else "<"
                        col_conds.append(f"{col} {op} {upper_str}")

                    if col_conds:
                        sql_conditions.append(" AND ".join(col_conds))

        return " AND ".join(
            f"({cond})" if "AND" in cond else cond for cond in sql_conditions
        )

    def extract_segments(self, df: pd.DataFrame) -> pd.DataFrame:
        """Iteratively extracts the highest-lift segments while enforcing exclusivity restrictions."""
        if self.enable_diversity:
            self._validate_feature_groups(df)

        current_df = df.copy()

        for i in range(1, self.max_segments + 1):
            base_rate = current_df[self.target].mean()
            if base_rate == 0 or len(current_df) < self.min_sample_size:
                break

            print(
                f"\n--- Iteration {i} | Remaining Rows: {len(current_df)} | Base Rate: {base_rate*100:.2f}% ---"
            )

            iv_ranking = self.compute_iv_ranking(current_df)
            top_vars = iv_ranking.head(self.top_n_vars)["variable"].tolist()
            binned_df = self.create_binned_df(current_df, top_vars)

            # FIX: Drop variables that failed to split (only 1 unique bin).
            # This completely stops uninformative (-inf, inf) ranges from masking as 2-way rules.
            valid_vars = [v for v in top_vars if binned_df[v].nunique() > 1]

            all_rules = []

            # ==========================================
            # APRIORI LEVEL 1 (Base Features)
            # ==========================================
            combos_1 = [(c,) for c in valid_vars]
            res_1 = self._agg_combinations(binned_df, combos_1, base_rate)

            valid_1way_vars = set()
            if not res_1.empty:
                valid_1way_vars = {c[0] for c in res_1["combo_vars"]}
                if self.enable_1way:
                    all_rules.append(res_1)

            if not valid_1way_vars:
                print(
                    "No base variables met the criteria. Stopping iterative extraction."
                )
                break

            # ==========================================
            # APRIORI LEVEL 2 (Pairs)
            # ==========================================
            valid_2way_sets = set()
            if len(valid_1way_vars) >= 2 and (self.enable_2way or self.enable_3way):
                combos_2 = [
                    c for c in combinations(valid_1way_vars, 2) if self.is_diverse(c)
                ]

                if combos_2:
                    res_2 = self._agg_combinations(binned_df, combos_2, base_rate)
                    if not res_2.empty:
                        valid_2way_sets = {frozenset(c) for c in res_2["combo_vars"]}
                        if self.enable_2way:
                            all_rules.append(res_2)

            # ==========================================
            # APRIORI LEVEL 3 (Triplets)
            # ==========================================
            if self.enable_3way and len(valid_1way_vars) >= 3 and valid_2way_sets:
                combos_3 = []
                for combo in combinations(valid_1way_vars, 3):
                    if not self.is_diverse(combo):
                        continue

                    pairs = [frozenset(p) for p in combinations(combo, 2)]
                    if all(p in valid_2way_sets for p in pairs):
                        combos_3.append(combo)

                if combos_3:
                    res_3 = self._agg_combinations(binned_df, combos_3, base_rate)
                    if not res_3.empty:
                        all_rules.append(res_3)

            # ==========================================
            # RULE SELECTION & POPULATION FILTERING
            # ==========================================
            if not all_rules:
                print(
                    "No active rules passed the criteria in this iteration. Stopping."
                )
                break

            shortlisted = pd.concat(all_rules, ignore_index=True)
            shortlisted = shortlisted.sort_values(
                ["lift", "rate", "count"], ascending=False
            ).reset_index(drop=True)

            best_rule = shortlisted.loc[0, "rule"]
            best_sql = self.parse_rule_to_sql(best_rule)

            self.segments.append(
                {
                    "segment_id": i,
                    "rule_string": best_rule,
                    "sql_filter": best_sql,
                    "count": shortlisted.loc[0, "count"],
                    "rate": shortlisted.loc[0, "rate"],
                    "lift": shortlisted.loc[0, "lift"],
                }
            )

            print(f"Segment {i} Found: {best_sql}")
            current_df = duckdb.query(
                f"SELECT * FROM current_df WHERE NOT ({best_sql})"
            ).df()

        return pd.DataFrame(self.segments).drop(columns=["combo_vars"], errors="ignore")

    def evaluate_final_coverage(self, original_df: pd.DataFrame) -> pd.DataFrame:
        """Compiles the sequential segment rules into a full CASE WHEN SQL query to map coverage."""
        if not self.segments:
            return pd.DataFrame()

        case_statements = [
            f"WHEN {seg['sql_filter']} THEN {seg['segment_id']}"
            for seg in self.segments
        ]
        case_sql = "\n                ".join(case_statements)

        final_query = f"""
        WITH PER_SEG_KPIS AS (
        SELECT 
            CASE 
                {case_sql}
                ELSE 0 
            END AS segment, 
            COUNT(*) AS total_count,
            SUM("{self.target}") AS target_events,
            (SUM("{self.target}") * 100.0 / COUNT(*)) AS response_rate
        FROM original_df
        GROUP BY 1),
        BASE_KPIS AS (
        SELECT *,
            SUM(total_count) OVER() AS total_population,
          (SUM(target_events) OVER()/SUM(total_count) OVER())*100 AS base_response_rate 
          FROM PER_SEG_KPIS
   )
          SELECT PER_SEG_KPIS.*, BASE_KPIS.base_response_rate,
            (PER_SEG_KPIS.total_count / BASE_KPIS.total_population)*100 AS capture_rate,
          (PER_SEG_KPIS.response_rate / BASE_KPIS.base_response_rate) AS lift
          FROM PER_SEG_KPIS
          LEFT JOIN BASE_KPIS
          ON PER_SEG_KPIS.segment = BASE_KPIS.segment
        ORDER BY segment
        """
        return duckdb.query(final_query).df()


# Configure logging for production tracking
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


class StrategicSegmentScore:
    """
    A high-performance engine using DuckDB to calculate weights using harmonic mean and lift,
    and extract decile boundaries over millions of rows in seconds.
    """

    def __init__(self, target_col: str, primary_key: str, segment_cols: List[str]):
        """
        :param target_col: The binary target column (1 = Event, 0 = Non-Event)
        :param primary_key: The unique identifier for the customer/row
        :param segment_cols: List of columns to use for segmenting
        """
        self.target_col = target_col
        self.primary_key = primary_key
        self.segment_cols = segment_cols
        self.model_artifact = {}

    def calculate_and_export_weights(
        self, df: pd.DataFrame, export_path: str = "scorecard_model.json"
    ) -> dict:
        """
        Calculates weights and decile thresholds using vectorized DuckDB execution,
        then exports the complete configuration to a JSON file.
        """
        logging.info(f"Initializing DuckDB processing for {len(df):,} rows...")

        # Connect to an in-memory DuckDB instance
        ctx = duckdb.connect()

        # Calculate baseline metrics across the entire dataset instantly
        baseline_res = ctx.execute(f"""
            SELECT 
                COUNT(*) as total_pop, 
                SUM({self.target_col}) as total_ev 
            FROM df
        """).fetchone()

        total_population = baseline_res[0]
        total_events = baseline_res[1]

        if total_population == 0 or total_events == 0:
            raise ValueError(
                "Invalid dataframe: Population or total events cannot be zero."
            )

        baseline_rate = total_events / total_population if total_population > 0 else 0
        zero_inflation_rate = 1 - baseline_rate
        segment_weights = {}

        # 1. Vectorized Segment Metric Calculation
        logging.info("Computing segment weights...")
        weights_lookup = {}
        for seg_col in self.segment_cols:
            # Drop into DuckDB to aggregate the specific segment instantly
            seg_res = ctx.execute(f"""
                SELECT 
                    COUNT(*) as seg_count,
                    SUM({self.target_col}) as seg_events
                FROM df 
                WHERE {seg_col} = 1
            """).fetchone()

            seg_count = seg_res[0] or 0
            seg_events = seg_res[1] or 0

            if seg_count == 0 or seg_events == 0:
                logging.warning(
                    f"Segment '{seg_col}' has 0 volume or events. Skipping."
                )
                continue

            response_rate = seg_events / seg_count
            capture_rate = seg_events / total_events
            lift = response_rate / baseline_rate

            # Harmonic mean calculation to balance response and capture rates
            if response_rate + capture_rate > 0:
                harmonic_mean = 2 * (
                    (response_rate * capture_rate) / (response_rate + capture_rate)
                )
                raw_weight = lift * harmonic_mean * 100
            else:
                raw_weight = 0

            weights_lookup[seg_col] = {
                "weight": int(np.round(raw_weight)),
                "lift": round(lift, 4),
                "response_rate": round(response_rate, 4),
                "capture_rate": round(capture_rate, 4),
            }

        # --- FIX A CORE LOGIC: ISOLATE ACTIVE POPULATION ---
        logging.info("Scoring training dataframe in-memory...")
        train_scores = np.zeros(total_population)
        for seg_col, meta in weights_lookup.items():
            train_scores += df[seg_col].values * meta["weight"]

        # Filter strictly for customers who hit at least one rule (> 0) only if zero-inflation is low(>=20%); otherwise, include all scores for decile calibration.
        logging.info(f"Dataset Zero-Inflation Rate: {zero_inflation_rate:.2%}")

        # Step 3: Route to the correct Deciling Methodology
        if zero_inflation_rate >= 0.80:
            logging.info(
                "High Zero-Inflation detected (>=80%). Applying Active-Only Deciling..."
            )
            active_scores = train_scores[train_scores > 0]
        else:
            active_scores = train_scores
            logging.info(
                "Low Zero-Inflation detected (<80%). Applying Full Population Deciling..."
            )

        if len(active_scores) == 0:
            raise ValueError("Model Failure: 0 customers triggered any segment rules.")

        logging.info(
            f"Calibrating deciles across {len(active_scores):,} active customers..."
        )
        s_scores = (
            pd.Series(active_scores).sort_values(ascending=False).reset_index(drop=True)
        )
        active_pop_size = len(s_scores)

        decile_thresholds = {}
        for d in range(1, 11):
            row_idx = int((d / 10.0) * active_pop_size) - 1
            row_idx = max(0, min(active_pop_size - 1, row_idx))
            decile_thresholds[str(d)] = int(s_scores.iloc[row_idx])

        self.model_artifact = {
            "model_metadata": {
                "total_training_population": int(total_population),
                "active_scored_population": int(active_pop_size),
                "active_population_pct": round(
                    (active_pop_size / total_population) * 100, 2
                ),
                "baseline_event_rate": round(baseline_rate, 4),
            },
            "segment_weights": weights_lookup,
            "decile_min_thresholds": decile_thresholds,
        }

        with open(export_path, "w") as f:
            json.dump(self.model_artifact, f, indent=4)

        return self.model_artifact


# Example Feature Groupings Setup
# business_groups = {
# "delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m","max_dpd_12m"],
# "transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
# "spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
# "repayment_vars":["payment_ratio_avg_3m","payment_ratio_avg_6m","payment_ratio_avg_12m"],
# "card_utilization_vars":["utilization_avg_3m","utilization_avg_6m","utilization_avg_12m","utilization_max_12m"],
# }

# Initialize the builder
builder = StrategicSegmentBuilder(
    target="default_flag",
    n_jobs=-1,
    min_sample_size=100,
    min_lift=1.0,
    top_n_vars=30,
    max_segments=5,
    enable_diversity=False,
    enable_1way=False,  # Exclude 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,
    feature_groups=None,
)

data = pd.read_csv(r"all_categorical_credit_default_50000.csv")

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)


--- Iteration 1 | Remaining Rows: 50000 | Base Rate: 14.77% ---
Segment 1 Found: region IN ('Tier2') AND risk_segment IN ('HighRisk') AND payment_band IN ('Missed')

--- Iteration 2 | Remaining Rows: 49888 | Base Rate: 14.58% ---
Segment 2 Found: risk_segment IN ('HighRisk') AND payment_band IN ('Missed') AND cash_advance_band IS NULL

--- Iteration 3 | Remaining Rows: 49778 | Base Rate: 14.40% ---
Segment 3 Found: relationship_depth IN ('MultiProduct', 'Premium') AND risk_segment IN ('HighRisk') AND payment_band IN ('Missed')

--- Iteration 4 | Remaining Rows: 49605 | Base Rate: 14.11% ---
Segment 4 Found: delinquency_band IN ('60Plus') AND region IN ('Rural') AND risk_segment IN ('SubPrime')

--- Iteration 5 | Remaining Rows: 49497 | Base Rate: 13.95% ---
Segment 5 Found: risk_segment IN ('HighRisk') AND occupation IN ('Student') AND utilization_band IN ('VeryHigh')


In [28]:
table = PrettyTable()
table.field_names = list(final_eval.columns)
for _, row in final_eval.iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+-------------------+--------------------+---------------------+--------------------+
| segment | total_count | target_events |   response_rate   | base_response_rate |     capture_rate    |        lift        |
+---------+-------------+---------------+-------------------+--------------------+---------------------+--------------------+
|   0.0   |   49360.0   |     6787.0    |       13.75       |       14.768       |        98.72        | 0.9310671722643553 |
|   1.0   |    112.0    |     110.0     | 98.21428571428571 |       14.768       | 0.22399999999999998 | 6.650479801888252  |
|   2.0   |    110.0    |     107.0     | 97.27272727272727 |       14.768       |         0.22        |  6.58672313601891  |
|   3.0   |    173.0    |     166.0     | 95.95375722543352 |       14.768       |        0.346        | 6.497410429674534  |
|   4.0   |    108.0    |      98.0     | 90.74074074074075 |       14.768       |        0.216        | 6.14441635568

In [29]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in segments_df.iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   region=<StringArray>
['Tier2']
Length: 1, dtype: str & risk_segment=<StringArray>
['HighRisk']
Length: 1, dtype: str & payment_band=<StringArray>
['Missed']
Length: 1, dtype: str
SQL Filter: region IN ('Tier2') AND risk_segment IN ('HighRisk') AND payment_band IN ('Missed')
--------------------------------------------------
Segment ID: 2
Raw Rule:   risk_segment=<StringArray>
['HighRisk']
Length: 1, dtype: str & payment_band=<StringArray>
['Missed']
Length: 1, dtype: str & cash_advance_band=Missing
SQL Filter: risk_segment IN ('HighRisk') AND payment_band IN ('Missed') AND cash_advance_band IS NULL
--------------------------------------------------
Segment ID: 3
Raw Rule:   relationship_depth=<StringArray>
['MultiProduct', 'Premium']
Length: 2, dtype: str & risk_segment=<StringArray>
['HighRisk']
Length: 1, dtype: str & payment_band=<StringArray>
['Missed']
Length: 1, dtype: str
SQL Filter: relationship_depth IN ('MultiProduct', 'Pr

In [30]:
final_eval

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,49360,6787.0,13.750000,14.768,98.720,0.931067
1,1,112,110.0,98.214286,14.768,0.224,6.650480
2,2,110,107.0,97.272727,14.768,0.220,6.586723
3,3,173,166.0,95.953757,14.768,0.346,6.497410
4,4,108,98.0,90.740741,14.768,0.216,6.144416
5,5,137,116.0,84.671533,14.768,0.274,5.733446


In [ ]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
    SELECT *, 
    CASE WHEN region IN ('Tier2') AND payment_band IN ('Missed') AND risk_segment IN ('HighRisk') THEN 1 ELSE 0 END AS seg_1,
    CASE WHEN payment_band IN ('Missed') AND risk_segment IN ('HighRisk') AND cash_advance_band IS NULL THEN 1 ELSE 0 END AS seg_2,
    CASE WHEN payment_band IN ('Missed') AND risk_segment IN ('HighRisk') AND relationship_depth IN ('MultiProduct', 'Premium') THEN 1 ELSE 0 END AS seg_3,
    CASE WHEN region IN ('Rural') AND risk_segment IN ('SubPrime') AND delinquency_band IN ('60Plus') THEN 1 ELSE 0 END AS seg_4,
    CASE WHEN risk_segment IN ('HighRisk') AND utilization_band IN ('VeryHigh') AND occupation IN ('Student') THEN 1 ELSE 0 END AS seg_5,
    CASE WHEN region IN ('Tier2') AND payment_band IN ('Missed') AND risk_segment IN ('HighRisk') THEN 1
    WHEN payment_band IN ('Missed') AND risk_segment IN ('HighRisk') AND cash_advance_band IS NULL THEN 2
    WHEN payment_band IN ('Missed') AND risk_segment IN ('HighRisk') AND relationship_depth IN ('MultiProduct', 'Premium') THEN 3
    WHEN region IN ('Rural') AND risk_segment IN ('SubPrime') AND delinquency_band IN ('60Plus') THEN 4
    WHEN risk_segment IN ('HighRisk') AND utilization_band IN ('VeryHigh') AND occupation IN ('Student') THEN 5
    ELSE 0 END AS segment_id
    FROM predicted
""").df()
conn.close()

In [32]:
predicted.reset_index(inplace=True)

In [ ]:
scorer = StrategicSegmentScore(
    target_col="default_flag",
    primary_key="index",
    segment_cols=["seg_1", "seg_2", "seg_3", "seg_4", "seg_5"],
)

In [ ]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test.json")

2026-06-28 03:20:22,903 - INFO - Initializing DuckDB processing for 50,000 rows...
2026-06-28 03:20:22,958 - INFO - Computing segment weights...
2026-06-28 03:20:23,160 - INFO - Scoring training dataframe in-memory...
2026-06-28 03:20:23,162 - INFO - Dataset Zero-Inflation Rate: 85.23%
2026-06-28 03:20:23,163 - INFO - High Zero-Inflation detected (>=20%). Applying Active-Only Deciling...
2026-06-28 03:20:23,164 - INFO - Calibrating deciles across 640 active customers...


{'model_metadata': {'total_training_population': 50000,
  'active_scored_population': 640,
  'active_population_pct': 1.28,
  'baseline_event_rate': 0.1477},
 'segment_weights': {'seg_1': {'weight': 20,
   'lift': 6.6505,
   'response_rate': 0.9821,
   'capture_rate': 0.0149},
  'seg_2': {'weight': 23,
   'lift': 6.5243,
   'response_rate': 0.9635,
   'capture_rate': 0.0179},
  'seg_3': {'weight': 54,
   'lift': 6.5637,
   'response_rate': 0.9693,
   'capture_rate': 0.0428},
  'seg_4': {'weight': 16,
   'lift': 6.1444,
   'response_rate': 0.9074,
   'capture_rate': 0.0133},
  'seg_5': {'weight': 19,
   'lift': 5.7974,
   'response_rate': 0.8562,
   'capture_rate': 0.0169}},
 'decile_min_thresholds': {'1': 77,
  '2': 74,
  '3': 54,
  '4': 54,
  '5': 54,
  '6': 20,
  '7': 19,
  '8': 19,
  '9': 16,
  '10': 16}}

In [ ]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/scorecard_model_test.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-06-28 03:20:23,194 - INFO - Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/scorecard_model_test.json...


In [ ]:
model_artifact.get("decile_min_thresholds")

{'1': 77,
 '2': 74,
 '3': 54,
 '4': 54,
 '5': 54,
 '6': 20,
 '7': 19,
 '8': 19,
 '9': 16,
 '10': 16}

In [37]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 20
Segment: seg_2 | Weight: 23
Segment: seg_3 | Weight: 54
Segment: seg_4 | Weight: 16
Segment: seg_5 | Weight: 19


In [ ]:
model_artifact.get("decile_min_thresholds")

{'1': 77,
 '2': 74,
 '3': 54,
 '4': 54,
 '5': 54,
 '6': 20,
 '7': 19,
 '8': 19,
 '9': 16,
 '10': 16}

In [51]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 20 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 23 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 54 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 16 ELSE 0 END AS seg_4_weighted,
    CASE WHEN seg_5 = 1 THEN 19 ELSE 0 END AS seg_5_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted + seg_5_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=77 THEN 1
                    WHEN total_weight >= 74 THEN 2
                    WHEN total_weight >= 54 THEN 3
                    WHEN total_weight >= 54 THEN 4
                    WHEN total_weight >= 54 THEN 5
                    WHEN total_weight >= 20 THEN 6
                    WHEN total_weight >= 19 THEN 7
                    WHEN total_weight >= 19 THEN 8
                    WHEN total_weight >= 19 THEN 9
                    WHEN total_weight >= 16 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [53]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(default_flag) AS events, 
                    (SUM(default_flag)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [54]:
scored

,decile_band,count,events,response_rate
0,0,49360,6787.0,13.750000
1,1,96,93.0,96.875000
2,2,57,57.0,100.000000
3,3,174,167.0,95.977011
4,6,68,66.0,97.058824
5,7,137,116.0,84.671533
6,10,108,98.0,90.740741
